# N083 · 并查集与动态连通性

**目标：** 只维护等价类信息并识别不支持的操作。

**先修：** N079, N080, N014。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** find/union；路径压缩；按大小合并；均摊；连通块计数。

## 从问题到算法

并查集维护等价类，而不是完整图。find返回连通分量代表，union仅在代表不同的时候连接，重复union不能减少分量数。按大小挂接控制树高，路径压缩缩短后续查找路径。账户合并以共享邮箱为连接证据，同名本身不足以证明同一账户。

## 最小实现

**本章接口：** `DSU`、`find_redundant_connection(edges)`、`accounts_merge(accounts)`

In [1]:
class DSU:
    def __init__(self,n): self.parent=list(range(n)); self.size=[1]*n; self.components=n
    def find(self,x):
        while self.parent[x]!=x:
            self.parent[x]=self.parent[self.parent[x]]; x=self.parent[x]
        return x
    def union(self,a,b):
        a=self.find(a); b=self.find(b)
        if a==b: return False
        if self.size[a]<self.size[b]: a,b=b,a
        self.parent[b]=a; self.size[a]+=self.size[b]; self.components-=1
        return True

def find_redundant_connection(edges):
    dsu=DSU(max((max(e) for e in edges),default=0)+1); answer=[]
    for u,v in edges:
        if not dsu.union(u,v): answer=[u,v]
    return answer

def accounts_merge(accounts):
    dsu=DSU(len(accounts)); owner={}
    for i,account in enumerate(accounts):
        for email in account[1:]:
            if email in owner: dsu.union(i,owner[email])
            else: owner[email]=i
    groups={}
    for i,account in enumerate(accounts): groups.setdefault(dsu.find(i),set()).update(account[1:])
    return [[accounts[root][0],*sorted(emails)] for root,emails in sorted(groups.items())]

## 正确性、前提与复杂度

代表不同表示当前没有连接路径，新边连接两个分量；代表相同表示边闭合了一条已有路径。压缩只把节点改挂同一根，不改变分量划分。

**代价：** 按大小合并加路径压缩的m次操作总O(mα(n))均摊界，空间O(n)。账户结果还需邮箱排序。普通DSU不支持任意删除边。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

dsu=DSU(5); rows=[]
for u,v in [(0,1),(1,2),(3,4),(0,2),(2,4)]:
    changed=dsu.union(u,v); rows.append(((u,v),changed,dsu.components,[dsu.find(i) for i in range(5)]))
show_table(['合并','是否新连接','分量数','各顶点代表'],rows)

合并,是否新连接,分量数,各顶点代表
"(0, 1)",True,4,"[0, 0, 2, 3, 4]"
"(1, 2)",True,3,"[0, 0, 0, 3, 4]"
"(3, 4)",True,2,"[0, 0, 0, 3, 3]"
"(0, 2)",False,2,"[0, 0, 0, 3, 3]"
"(2, 4)",True,1,"[0, 0, 0, 0, 0]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
d=DSU(3); assert d.union(0,1) and not d.union(1,0) and d.components==2
assert find_redundant_connection([[1,2],[1,3],[2,3]])==[2,3]
accounts=[['John','a@x','b@x'],['John','b@x','c@x'],['John','z@x']]
out=accounts_merge(accounts); assert sorted(out)==[['John','a@x','b@x','c@x'],['John','z@x']]
from random import Random
rng=Random(83); d=DSU(8); adj=[set() for _ in range(8)]
for _ in range(100):
    u,v=rng.randrange(8),rng.randrange(8); d.union(u,v); adj[u].add(v); adj[v].add(u)
    for s in range(8):
        seen={s}; stack=[s]
        while stack:
            for t in adj[stack.pop()]:
                if t not in seen: seen.add(t); stack.append(t)
        assert all((d.find(s)==d.find(t))==(t in seen) for t in range(8))
print('N083: 所有本章断言通过')

N083: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 对比连通性与路径查询。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 解释普通DSU不能直接删除边。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 684. Redundant Connection（canonical）
- 721. Accounts Merge（canonical）
- 1319. Number of Operations to Make Network Connected（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。